# Embeddings, Vector Databases, and Search

Converting text into embedding vectors is the first step to any text processing pipeline. As the amount of text gets larger, there is often a need to save these embedding vectors into a dedicated vector index or library, so that developers won't have to recompute the embeddings and the retrieval process is faster. We can then search for documents based on our intended query and pass these relevant documents into a language model (LM) as additional context. We also refer to this context as supplying the LM with "state" or "memory". The LM then generates a response based on the additional context it receives! 

In this notebook, we will implement the full workflow of text vectorization, vector search, and question answering workflow. While we use [FAISS](https://faiss.ai/) (vector library) and [ChromaDB](https://docs.trychroma.com/) (vector database), and a Hugging Face model, know that you can easily swap these tools out for your preferred tools or models!

<img src="https://files.training.databricks.com/images/llm/updated_vector_search.png" width=1000 target="_blank" > 

![Dolly](https://files.training.databricks.com/images/llm/dolly_small.png) 

### Learning Objectives
1. Implement the workflow of reading text, converting text to embeddings, saving them to FAISS and ChromaDB 
2. Query for similar documents using FAISS and ChromaDB 
3. Apply a Hugging Face language model for question answering!

### Libraries

`faiss-cpu==1.7.4 chromadb==0.3.21`


Note that the chromadb API and data format(s) have changed from 0.3.21 so this notebook will not run without upgrading it.

In [1]:
from notebook_helper import (
    start_execution_time,
    execution_time,
)

start_time = start_execution_time()

Notebook execution start time: 06-12-2023 14:38:18


## Step 1: Reading data

In this section, we are going to use the data on <a href="https://newscatcherapi.com/" target="_blank">news topics collected by the NewsCatcher team</a>, who collect and index news articles and release them to the open-source community. The dataset can be downloaded from <a href="https://www.kaggle.com/kotartemiy/topic-labeled-news-dataset" target="_blank">Kaggle</a>.


In [2]:
import pandas as pd

from sentence_transformers import InputExample
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


import chromadb
from chromadb.config import Settings
import json

/Users/mjboothaus/code/github/mjboothaus/large-language-models-da/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)  # for displaying dataframes in a more friendly manner

<IPython.core.display.Javascript object>

In [4]:
DA_paths_datasets = "../data"  # note rename of DA.paths.datasets variable (for local Jupyter notebook versions)
DA_paths_user_db = "../data/chroma"

In [5]:
pdf = pd.read_csv(f"{DA_paths_datasets}/news/labelled_newscatcher_dataset.csv", sep=";")
pdf["id"] = pdf.index
display(pdf)

topic                                               link  \
0             SCIENCE  https://www.eurekalert.org/pub_releases/2020-0...   
1             SCIENCE  https://www.pulse.ng/news/world/an-irresistibl...   
2             SCIENCE  https://www.express.co.uk/news/science/1322607...   
3             SCIENCE  https://www.ndtv.com/world-news/glaciers-could...   
4             SCIENCE  https://www.thesun.ie/tech/5742187/perseid-met...   
...               ...                                                ...   
108769         NATION  https://www.vanguardngr.com/2020/08/pdp-govern...   
108770       BUSINESS  https://www.patentlyapple.com/patently-apple/2...   
108771         HEALTH  https://www.belfastlive.co.uk/news/health/coro...   
108772  ENTERTAINMENT  https://www.thenews.com.pk/latest/696364-paul-...   
108773         SPORTS  https://www.balls.ie/football/shane-duffy-brig...   

                   domain       published_date  \
0          eurekalert.org  2020-08-06 13:59:45   
1                pulse.ng  2020-08-12 15:14:19   
2           express.co.uk  2020-08-13 21:01:00   
3                ndtv.com  2020-08-03 22:18:26   
4               thesun.ie  2020-08-12 19:54:36   
...                   ...                  ...   
108769    vanguardngr.com  2020-08-08 02:40:00   
108770  patentlyapple.com  2020-08-08 01:27:12   
108771  belfastlive.co.uk  2020-08-12 17:01:00   
108772     thenews.com.pk  2020-08-05 04:59:00   
108773           balls.ie  2020-08-09 10:25:26   

                                                    title lang      id  
0       A closer look at water-splitting's solar fuel ...   en       0  
1       An irresistible scent makes locusts swarm, stu...   en       1  
2       Artificial intelligence warning: AI will know ...   en       2  
3        Glaciers Could Have Sculpted Mars Valleys: Study   en       3  
4       Perseid meteor shower 2020: What time and how ...   en       4  
...                                                   ...  ...     ...  
108769  PDP governors’ forum urges security agencies t...   en  108769  
108770  In Q2-20, Apple Dominated the Premium Smartpho...   en  108770  
108771  Coronavirus Northern Ireland: Full breakdown s...   en  108771  
108772  Paul McCartney details post-Beatles distress a...   en  108772  
108773  Report: Talks Underway To Keep Shane Duffy In ...   en  108773  

[108774 rows x 7 columns]

## Vector Library: FAISS

Vector libraries are often sufficient for small, static data. Since it's not a full-fledged database solution, it doesn't have the CRUD (Create, Read, Update, Delete) support. Once the index has been built, if there are more vectors that need to be added/removed/edited, the index has to be rebuilt from scratch. 

That said, vector libraries are easy, lightweight, and fast to use. Examples of vector libraries are [FAISS](https://faiss.ai/), [ScaNN](https://github.com/google-research/google-research/tree/master/scann), [ANNOY](https://github.com/spotify/annoy), and [HNSM](https://arxiv.org/abs/1603.09320).

FAISS has several ways for similarity search: L2 (Euclidean distance), cosine similarity. You can read more about their implementation on their [GitHub](https://github.com/facebookresearch/faiss/wiki/Getting-started#searching) page or [blog post](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/). They also published their own [best practice guide here](https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index).

If you'd like to read up more on the comparisons between vector libraries and databases, [here is a good blog post](https://weaviate.io/blog/vector-library-vs-vector-database#feature-comparison---library-versus-database).


The overall workflow of FAISS is captured in the diagram below. 

<img src="https://miro.medium.com/v2/resize:fit:1400/0*ouf0eyQskPeGWIGm" width=700>

Source: [How to use FAISS to build your first similarity search by Asna Shafiq](https://medium.com/loopio-tech/how-to-use-faiss-to-build-your-first-similarity-search-bf0f708aa772).


In [6]:
SUBSET_SIZE = 10000 # (originally 1000)

pdf_subset = pdf.head(SUBSET_SIZE)


def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text
    """
    return InputExample(texts=[doc1])


faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x["title"]), axis=1).tolist()

In [7]:
faiss_train_examples[0]

### Step 2: Vectorize text into embedding vectors

We will be using `Sentence-Transformers` [library](https://www.sbert.net/) to load a language model to vectorize our text into embeddings. The library hosts some of the most popular transformers on [Hugging Face Model Hub](https://huggingface.co/sentence-transformers).
Here, we are using the `model = SentenceTransformer("all-MiniLM-L6-v2")` to generate embeddings.



In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2", cache_folder=DA_paths_datasets)  # Use a pre-cached model
faiss_title_embedding = model.encode(pdf_subset.title.values.tolist())
len(faiss_title_embedding), len(faiss_title_embedding[0])

(10000, 384)

### Step 3: Saving embedding vectors to FAISS index
Below, we create the FAISS index object based on our embedding vectors, normalize vectors, and add these vectors to the FAISS index. 

In [9]:
pdf_to_index = pdf_subset.set_index(["id"], drop=False)
id_index = np.array(pdf_to_index.id.values).flatten().astype("int")

content_encoded_normalized = faiss_title_embedding.copy()
faiss.normalize_L2(content_encoded_normalized)

# Index1DMap translates search results to IDs: https://faiss.ai/cpp_api/file/IndexIDMap_8h.html#_CPPv4I0EN5faiss18IndexIDMapTemplateE
# The IndexFlatIP below builds index
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)

## Step 4: Search for relevant documents

We define a search function below to first vectorize our query text, and then search for the vectors with the closest distance. 


In [10]:
def search_content(query, pdf_to_index, k=3):
    query_vector = model.encode([query])
    faiss.normalize_L2(query_vector)

    # We set k to limit the number of vectors we want to return
    top_k = index_content.search(query_vector, k)
    ids = top_k[1][0].tolist()
    similarities = top_k[0][0].tolist()
    results = pdf_to_index.loc[ids]
    results["similarities"] = similarities
    return results

Tada! Now you can query for similar content! Notice that you did not have to configure any database networks beforehand nor pass in any credentials. FAISS works locally with your code.


In [11]:
display(search_content("animal", pdf_to_index))

topic                                               link  \
id                                                                    
3426     SCIENCE  https://www.nytimes.com/2020/08/12/science/rep...   
176   TECHNOLOGY  https://www.pushsquare.com/news/2020/08/random...   
7130      HEALTH  https://www.jpost.com/health-science/social-di...   

              domain       published_date  \
id                                          
3426     nytimes.com  2020-08-12 14:31:00   
176   pushsquare.com  2020-08-03 16:30:00   
7130       jpost.com  2020-08-15 20:56:00   

                                                  title lang    id  \
id                                                                   
3426  Making Sense of ‘One of the Most Baffling Anim...   en  3426   
176   Random: You Can Pick Up and Pet Cats in Assass...   en   176   
7130  Social distancing in animals sheds light on it...   en  7130   

      similarities  
id                  
3426      0.457315  
176       0.391902  
7130      0.390306

Up until now, we haven't done the last step of conducting Q/A with a language model yet. We are going to demonstrate this with Chroma, a vector database.

## Vector Database: Chroma

Chroma is an open-source embedding database. The company just raised its [seed funding in April 2023](https://www.trychroma.com/blog/seed) and is quickly becoming popular to support LLM-based applications. 


In [12]:
chroma_client = chromadb.Client(
    Settings(
        chroma_db_impl="duckdb+parquet",
        persist_directory=DA_paths_user_db,  # this is an optional argument. If you don't supply this, the data will be ephemeral
    )
)

### Chroma Concept: Collection

Chroma `collection` is akin to an index that stores one set of your documents. 

According to the [docs](https://docs.trychroma.com/getting-started): 
> Collections are where you will store your embeddings, documents, and additional metadata

The nice thing about ChromaDB is that if you don't supply a model to vectorize text into embeddings, it will automatically load a default embedding function, i.e. `SentenceTransformerEmbeddingFunction`. It can handle tokenization, embedding, and indexing automatically for you. If you would like to change the embedding model, read [here on how to do that](https://docs.trychroma.com/embeddings). TLDR: you can add an optional `model_name` argument. 

You can read [the documentation here](https://docs.trychroma.com/usage-guide#using-collections) on rules for collection names.


In [13]:
collection_name = "my_news"

# If you have created the collection before, you need to delete the collection first
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

Creating collection: 'my_news'


### Step 1: Add data to collection

Since we are re-using the same data, we can skip the step of reading data. As mentioned in the text above, Chroma can take care of text vectorization for us, so we can directly add text to the collection and Chroma will convert the text into embeddings behind the scene. 


In [14]:
display(pdf_subset)

topic                                               link  \
0        SCIENCE  https://www.eurekalert.org/pub_releases/2020-0...   
1        SCIENCE  https://www.pulse.ng/news/world/an-irresistibl...   
2        SCIENCE  https://www.express.co.uk/news/science/1322607...   
3        SCIENCE  https://www.ndtv.com/world-news/glaciers-could...   
4        SCIENCE  https://www.thesun.ie/tech/5742187/perseid-met...   
...          ...                                                ...   
9995  TECHNOLOGY  https://www.cnet.com/news/best-google-assistan...   
9996  TECHNOLOGY  https://www.somagnews.com/scientists-discover-...   
9997  TECHNOLOGY  https://www.forbes.com/sites/jaymcgregor/2020/...   
9998  TECHNOLOGY  https://au.finance.yahoo.com/news/spacex-attem...   
9999       WORLD  https://www.dailymail.co.uk/news/article-86314...   

                    domain       published_date  \
0           eurekalert.org  2020-08-06 13:59:45   
1                 pulse.ng  2020-08-12 15:14:19   
2            express.co.uk  2020-08-13 21:01:00   
3                 ndtv.com  2020-08-03 22:18:26   
4                thesun.ie  2020-08-12 19:54:36   
...                    ...                  ...   
9995              cnet.com  2020-08-11 14:35:15   
9996         somagnews.com  2020-08-16 11:20:00   
9997            forbes.com  2020-08-10 12:00:00   
9998  au.finance.yahoo.com  2020-08-17 12:32:00   
9999       dailymail.co.uk  2020-08-15 21:51:49   

                                                  title lang    id  
0     A closer look at water-splitting's solar fuel ...   en     0  
1     An irresistible scent makes locusts swarm, stu...   en     1  
2     Artificial intelligence warning: AI will know ...   en     2  
3      Glaciers Could Have Sculpted Mars Valleys: Study   en     3  
4     Perseid meteor shower 2020: What time and how ...   en     4  
...                                                 ...  ...   ...  
9995  The best Google Assistant and Nest devices of ...   en  9995  
9996  Scientists Discover a Liquid that Acts Like a ...   en  9996  
9997  Samsung Note 20 Ultra Gains Gaming Advantage O...   en  9997  
9998  SpaceX will attempt to break a rocket reusabil...   en  9998  
9999  Heathrow Airport launches £150 coronavirus tes...   en  9999  

[10000 rows x 7 columns]

Each document must have a unique `id` associated with it and it is up to you to check that there are no duplicate ids. 

Adding data to collection will take some time to run, especially when there is a lot of data. In the cell below, we intentionally write only a subset of data to the collection to speed things up. 


In [15]:
collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[f"id{x}" for x in range(100)],
)

### Step 2: Query for 10 relevant documents on "space"

We will return 10 most relevant documents. You can think of `10` as 10 nearest neighbors. You can also change the number of results returned as well. 

In [16]:
results = collection.query(query_texts=["space"], n_results=10)

print(json.dumps(results, indent=4))

{
    "ids": [
        [
            "id72",
            "id7",
            "id30",
            "id26",
            "id23",
            "id76",
            "id69",
            "id40",
            "id47",
            "id75"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Beck teams up with NASA and AI for 'Hyperspace' visual album experience",
            "Orbital space tourism set for rebirth in 2021",
            "NASA drops \"insensitive\" nicknames for cosmic objects",
            "\u2018It came alive:\u2019 NASA astronauts describe experiencing splashdown in SpaceX Dragon",
            "Hubble Uses Moon As \u201cMirror\u201d to Study Earth\u2019s Atmosphere \u2013 Proxy in Search of Potentially Habitable Planets Around Other Stars",
            "Australia's small yet crucial part in the mission to find life on Mars",
            "NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico",
            "SpaceX's Starship spacecraft saw 150 mete

### Bonus: Add filter statement

In addition to conducting relevancy search, we can also add filter statements. Refer to the [documentation](https://docs.trychroma.com/usage-guide#using-where-filters) for more information.


In [17]:
collection.query(query_texts=["space"], where={"topic": "SCIENCE"}, n_results=10)

{'ids': [['id7',
   'id30',
   'id26',
   'id23',
   'id76',
   'id69',
   'id40',
   'id47',
   'id75',
   'id52']],
 'embeddings': None,
 'documents': [['Orbital space tourism set for rebirth in 2021',
   'NASA drops "insensitive" nicknames for cosmic objects',
   '‘It came alive:’ NASA astronauts describe experiencing splashdown in SpaceX Dragon',
   'Hubble Uses Moon As “Mirror” to Study Earth’s Atmosphere – Proxy in Search of Potentially Habitable Planets Around Other Stars',
   "Australia's small yet crucial part in the mission to find life on Mars",
   'NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico',
   "SpaceX's Starship spacecraft saw 150 meters high",
   'NASA’s InSight lander shows what’s beneath Mars’ surface',
   'Alien base on Mercury: ET hunters claim to find huge UFO',
   'SpaceX Crew-1 mission with NASA, first fully operational crewed mission to space to launch in October']],
 'metadatas': [[{'topic': 'SCIENCE'},
   {'topic': 'SCIENCE'},
   {'topic': '

### Bonus: Update data in a collection

Unlike a vector library, vector databases support changes to the data so we can update or delete data. 

Indeed, we can update or delete data in a Chroma collection. 


In [18]:
collection.delete(ids=["id0"])

[UUID('30340a02-090e-4afa-bbcd-b6890a90048f')]

The record with `ids=0` is no longer present.

In [19]:
collection.get(
    ids=["id0"],
)

{'ids': [], 'embeddings': None, 'documents': [], 'metadatas': []}

We can also update a specific data point.


In [20]:
collection.get(
    ids=["id2"],
)


collection.update(
    ids=["id2"],
    metadatas=[{"topic": "TECHNOLOGY"}],
)

## Prompt engineering for question answering 

Now that we have identified documents about space from the news dataset, we can pass these documents as additional context for a language model to generate a response based on them! 

We first need to pick a `text-generation` model. Below, we use a Hugging Face model. You can also use OpenAI as well, but you will need to get an Open AI token and [pay based on the number of tokens](https://openai.com/pricing). 


In [21]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=DA_paths_datasets)
lm_model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=DA_paths_datasets)

pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    device_map="auto",
)

Here's where prompt engineering, which is developing prompts, comes in. We pass in the context in our `prompt_template` but there are numerous ways to write a prompt. Some prompts may generate better results than the others and it requires some experimentation to figure out how best to talk to the model. Each language model behaves differently to prompts. 

Our prompt template below is inspired from a [2023 paper on program-aided language model](https://arxiv.org/pdf/2211.10435.pdf). The authors have provided their sample prompt template [here](https://github.com/reasoning-machines/pal/blob/main/pal/prompt/date_understanding_prompt.py).

The following links also provide some helpful guidance on prompt engineering: 
- [Prompt engineering with OpenAI](https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering-with-openai-api)
- [GitHub repo that compiles best practices to interact with ChatGPT](https://github.com/f/awesome-chatgpt-prompts)


In [22]:
question = "What's the latest news on space development?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"


lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Relevant context: #Beck teams up with NASA and AI for 'Hyperspace' visual album experience #Orbital space tourism set for rebirth in 2021 #NASA drops "insensitive" nicknames for cosmic objects #‘It came alive:’ NASA astronauts describe experiencing splashdown in SpaceX Dragon #Hubble Uses Moon As “Mirror” to Study Earth’s Atmosphere – Proxy in Search of Potentially Habitable Planets Around Other Stars #Australia's small yet crucial part in the mission to find life on Mars #NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico #SpaceX's Starship spacecraft saw 150 meters high #NASA’s InSight lander shows what’s beneath Mars’ surface #Alien base on Mercury: ET hunters claim to find huge UFO

 The user's question: What's the latest news on space development? And what the latest headlines mean to you?

So what is the current state of space exploration?

‪This post is sponsored by NASA!‪

‪As mentioned before, this post is sponsored by NASA (SpaceX). You can follow and/or support S

Yay, you have just completed the implementation of your first text vectorization, search, and question answering workflow (that requires prompt engineering)!

In the lab, you will apply your newly gained knowledge to a different dataset. You can also check out the optional modules on Pinecone and Weaviate to learn how to set up vector databases that offer enterprise offerings.

In [23]:
exec_time = execution_time(start_time)

Notebook execution finished --- Elapsed time 45 seconds ---
